# Sélectionner les voies pour un premier réseau piéton

Nous allons attribuer à chaque voie de la zone test un statut et une raison :

- `included` : retenue par notre profil de marche ;
- `excluded` : écartée selon une règle explicite ;
- `review` : à examiner avant de l'utiliser.

Ce **profil** est notre ensemble de règles pour un prototype de marche générale. Une voie retenue selon des valeurs par défaut reste une hypothèse : ce n'est pas une vérification de terrain, ni une garantie d'accessibilité aux personnes à mobilité réduite.

Ouvrir avec **Python (IGN / OSM)**. Le notebook 04 doit avoir produit l'échantillon enrichi avec les restrictions d'accès. Aucune nouvelle bibliothèque n'est nécessaire.

Sources pour les conventions OSM : [accès](https://wiki.openstreetmap.org/wiki/Key:access?uselang=nl), [restrictions et valeurs implicites](https://wiki.openstreetmap.org/wiki/Restrictions), [limites des profils par défaut](https://wiki.openstreetmap.org/wiki/Default_access_restrictions).

## 1. Charger l'échantillon

Nous reprenons le fichier exploré au notebook 04. Les restrictions plus spécifiques, comme `foot:conditional`, ont été conservées sous des noms anglais avec des underscores : `foot_conditional`.

Si le fichier provient d'une ancienne version du notebook 04 et qu'il manque des colonnes, relancer ce notebook avant de continuer. Une colonne manquante ne doit pas être traitée comme une absence de restriction dans OSM.

In [1]:
from pathlib import Path
import pandas as pd

import geopandas as gpd
from IPython.display import HTML, display

current_directory = Path.cwd().resolve()
module_directory = None

for directory in [current_directory, *current_directory.parents]:
    if directory.name == "ign-osm" and (directory / "requirements.txt").is_file():
        module_directory = directory
        break
    if (directory / "ign-osm" / "requirements.txt").is_file():
        module_directory = directory / "ign-osm"
        break

if module_directory is None:
    raise FileNotFoundError("Open the notebook from the project directory.")

input_path = module_directory / "data" / "processed" / "clermont_streets_sample.geojson"
audit_path = module_directory / "data" / "processed" / "clermont_walking_audit.geojson"
output_path = module_directory / "data" / "processed" / "clermont_walking_ways.geojson"
preview_path = module_directory / "data" / "processed" / "clermont_walking_preview.html"

streets = gpd.read_file(input_path)
required_columns = {
    "osm_way_id", "highway", "foot", "access", "sidewalk", "motorroad",
    "foot_conditional", "access_conditional", "foot_forward", "foot_backward",
    "oneway_foot", "opening_hours", "geometry",
}
missing_columns = required_columns - set(streets.columns)
if missing_columns:
    raise ValueError(f"Run the updated notebook 04 first. Missing columns: {sorted(missing_columns)}")
print(f"Input ways: {len(streets)}")

Input ways: 3736


## 2. Définir des règles lisibles

L'ordre des règles compte : le premier cas reconnu décide du statut.

| Cas | Décision du prototype |
|---|---|
| Voie en construction ou seulement proposée | Exclue |
| Horaires, restrictions conditionnelles ou directionnelles | À examiner |
| `foot=no` ou `foot=private` | Exclue |
| `foot=yes`, `designated` ou `permissive` | Retenue, sauf conflit avec un type de voie rapide |
| Autre valeur explicite de `foot` | À examiner |
| Sans `foot`, `access=no` ou `private` | Exclue |
| Sans `foot`, accès limité à une destination, des clients ou autre valeur inconnue | À examiner |
| Autoroute, voie rapide ou `motorroad=yes` sans autorisation piétonne explicite | Exclue par prudence |
| Trottoir signalé comme séparé, sans précision piétonne | À examiner |
| Rue ou chemin d'un type retenu, sans restriction connue | Retenu par hypothèse |
| Type non couvert, notamment piste cyclable seule ou ascenseur | À examiner |

`foot` prime sur la règle générale `access`. `permissive` signifie que le propriétaire tolère l'accès ; nous le retenons pour ce prototype, mais cette permission peut être retirée. Les accès réservés à une destination ne sont pas assimilés à du transit public.

Les escaliers sont retenus pour la marche générale. Ils devront être écartés ou pénalisés dans un profil adapté aux personnes ne pouvant pas les emprunter. Les choix par défaut ci-dessous sont des hypothèses du projet, pas une transcription exhaustive des règles locales.

In [2]:
default_walking_types = {
    "footway", "pedestrian", "steps", "path", "residential", "living_street",
    "service", "unclassified", "primary", "primary_link", "secondary",
    "secondary_link", "tertiary", "tertiary_link", "track",
}
fast_road_types = {"motorway", "motorway_link", "trunk", "trunk_link"}
allowed_values = {"yes", "designated", "permissive"}
restricted_values = {"no", "private"}
complex_access_columns = [
    "foot_conditional", "access_conditional", "foot_forward", "foot_backward",
    "oneway_foot", "opening_hours",
]


def classify_walking_access(row):
    highway = row.get("highway")
    foot = row.get("foot")
    access = row.get("access")
    fast_road = highway in fast_road_types or row.get("motorroad") == "yes"

    if highway in {"construction", "proposed"}:
        return "excluded", "inactive_way"
    if any(pd.notna(row.get(column)) for column in complex_access_columns):
        return "review", "conditional_directional_or_timed_access"
    if foot in restricted_values:
        return "excluded", "explicit_foot_restriction"
    if foot in allowed_values:
        if fast_road:
            return "review", "foot_permission_on_fast_road"
        return "included", "explicit_foot_permission"
    if pd.notna(foot):
        return "review", "other_foot_rule"
    if access in restricted_values:
        return "excluded", "general_access_restriction"
    if pd.notna(access) and access not in allowed_values:
        return "review", "limited_or_unknown_general_access"
    if fast_road:
        return "excluded", "fast_road_without_foot_permission"
    if row.get("sidewalk") == "separate":
        return "review", "separately_mapped_sidewalk"
    if highway in default_walking_types:
        return "included", "assumed_access_by_highway_type"
    return "review", "highway_type_needs_review"

## 3. Vérifier quelques cas importants

Ces exemples vérifient des interactions qui changent le résultat : une autorisation piétonne spécifique peut lever une restriction générale ; une rue résidentielle peut être privée ; une piste cyclable seule ne prouve pas un droit d'accès à pied.

Les exemples sont artificiels et complètent les contrôles des données réelles, sans constituer une validation juridique du profil.

In [3]:
examples = [
    {"highway": "residential", "expected": "included"},
    {"highway": "residential", "foot": "no", "expected": "excluded"},
    {"highway": "footway", "access": "private", "expected": "excluded"},
    {"highway": "footway", "access": "private", "foot": "yes", "expected": "included"},
    {"highway": "service", "access": "destination", "expected": "review"},
    {"highway": "cycleway", "expected": "review"},
    {"highway": "cycleway", "foot": "yes", "expected": "included"},
    {"highway": "motorway", "expected": "excluded"},
    {"highway": "primary", "motorroad": "yes", "expected": "excluded"},
    {"highway": "footway", "foot_conditional": "no @ (sunset-sunrise)", "expected": "review"},
]
example_results = []
for example in examples:
    status, reason = classify_walking_access(example)
    assert status == example["expected"], example
    example_results.append({**example, "actual": status, "reason": reason})
display(pd.DataFrame(example_results).fillna(""))

,highway,expected,actual,reason,foot,access,motorroad,foot_conditional
0,residential,included,included,assumed_access_by_highway_type,,,,
1,residential,excluded,excluded,explicit_foot_restriction,no,,,
2,footway,excluded,excluded,general_access_restriction,,private,,
3,footway,included,included,explicit_foot_permission,yes,private,,
4,service,review,review,limited_or_unknown_general_access,,destination,,
5,cycleway,review,review,highway_type_needs_review,,,,
6,cycleway,included,included,explicit_foot_permission,yes,,,
7,motorway,excluded,excluded,fast_road_without_foot_permission,,,,
8,primary,excluded,excluded,fast_road_without_foot_permission,,,yes,
9,footway,review,review,conditional_directional_or_timed_access,,,,no @ (sunset-sunrise)


## 4. Appliquer le profil aux données réelles

`apply(..., axis=1)` appelle notre fonction pour chaque ligne. Les deux valeurs retournées deviennent les colonnes `walking_status` et `walking_reason`.

Nous conservons toutes les voies dans un fichier d'audit. L'**audit** permet de retrouver pourquoi une voie a été retenue, exclue ou laissée à examiner. Le fichier des voies retenues contient uniquement le statut `included`.

In [4]:
decisions = streets.apply(classify_walking_access, axis=1, result_type="expand")
decisions.columns = ["walking_status", "walking_reason"]
classified_streets = streets.join(decisions)
walking_ways = classified_streets.loc[
    classified_streets["walking_status"] == "included"
].copy()

assert len(classified_streets) == len(streets)
assert classified_streets["walking_status"].isin({"included", "excluded", "review"}).all()
assert classified_streets["walking_reason"].notna().all()
assert not walking_ways.empty

display(classified_streets.groupby(["walking_status", "walking_reason"]).size().rename("way_count").to_frame())
display(classified_streets.loc[
    classified_streets["walking_status"] == "review",
    ["osm_way_id", "name", "highway", "foot", "access", "walking_reason"],
].head(15))

way_count
walking_status walking_reason                                    
excluded       explicit_foot_restriction                       16
               general_access_restriction                     175
               inactive_way                                     4
included       assumed_access_by_highway_type                3255
               explicit_foot_permission                       144
review         conditional_directional_or_timed_access          1
               highway_type_needs_review                       96
               limited_or_unknown_general_access               42
               separately_mapped_sidewalk                       3

,osm_way_id,name,highway,foot,access,walking_reason
8,4080631,Boulevard François Mitterrand,busway,NaN,NaN,highway_type_needs_review
22,4082393,Rue des Minimes,pedestrian,NaN,destination,limited_or_unknown_general_access
26,4082805,Cours Sablon,secondary,NaN,NaN,separately_mapped_sidewalk
40,4208588,Rue Cadene,pedestrian,NaN,destination,limited_or_unknown_general_access
41,4208589,Rue de la Forge,pedestrian,NaN,destination,limited_or_unknown_general_access
129,32124229,Rue de la Tannerie,pedestrian,NaN,destination,limited_or_unknown_general_access
136,32124242,Rue du Port,pedestrian,NaN,destination,limited_or_unknown_general_access
165,32434264,Rue Dallet,residential,NaN,destination,limited_or_unknown_general_access
217,34785776,Rue Terrasse,residential,NaN,destination,limited_or_unknown_general_access
243,37880115,Chemin de Beaurepaire,residential,NaN,destination,limited_or_unknown_general_access


## 5. Exporter et relire les résultats

Les fichiers sont des GeoJSON en longitude/latitude. Les sorties sont remplacées lors d'une nouvelle exécution, mais la source du notebook 04 reste intacte.

Nous relisons les deux sorties pour contrôler leurs identifiants et statuts.

In [5]:
classified_streets.to_file(audit_path, driver="GeoJSON")
walking_ways.to_file(output_path, driver="GeoJSON")
saved_audit = gpd.read_file(audit_path)
saved_walking_ways = gpd.read_file(output_path)
assert set(saved_audit["osm_way_id"]) == set(streets["osm_way_id"])
assert set(saved_walking_ways["osm_way_id"]) == set(walking_ways["osm_way_id"])
assert saved_walking_ways["walking_status"].eq("included").all()
assert saved_walking_ways.geometry.is_valid.all()
print(f"Saved {len(saved_audit)} audit ways: {audit_path}")
print(f"Saved {len(saved_walking_ways)} included ways: {output_path}")

Saved 3736 audit ways: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm/data/processed/clermont_walking_audit.geojson
Saved 3399 included ways: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm/data/processed/clermont_walking_ways.geojson


## 6. Comparer sur une carte

Les voies retenues sont en bleu, les voies exclues en gris et les cas à examiner en orange. La couleur correspond à notre décision de traitement, pas à un constat d'accessibilité sur le terrain.

L'aperçu est affiché ci-dessous et sauvegardé en HTML hors connexion. Le nord est en haut ; aucun fond de carte distant n'est utilisé.

In [6]:
preview_streets = saved_audit.to_crs(epsg=2154)
min_x, min_y, max_x, max_y = preview_streets.total_bounds
padding = 50
width = max_x - min_x + 2 * padding
height = max_y - min_y + 2 * padding
view_box = f"{min_x - padding} {-max_y - padding} {width} {height}"
colors = {"excluded": "#94a3b8", "review": "#ea580c", "included": "#2563eb"}
svg_groups = []
for status, color in colors.items():
    geometries = preview_streets.loc[preview_streets["walking_status"] == status, "geometry"]
    lines = "\n".join(geometry.svg() for geometry in geometries)
    svg_groups.append(f'<g style="--way-color: {color}">{lines}</g>')
street_svg = "\n".join(svg_groups)
counts = saved_audit["walking_status"].value_counts()

preview_html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Clermont-Ferrand walking access profile</title>
  <style>
    .walking-preview {{ font-family: sans-serif; max-width: 1000px; margin: 24px auto; padding: 16px; }}
    .walking-preview svg {{ width: 100%; height: 70vh; background: #f8fafc; }}
    .walking-preview polyline, .walking-preview path {{ fill: none; stroke: var(--way-color); stroke-width: 2; }}
  </style>
</head>
<body>
  <div class="walking-preview">
    <h1>Clermont-Ferrand: walking access profile</h1>
    <p>Blue: included ({counts.get('included', 0)}) |
       Grey: excluded ({counts.get('excluded', 0)}) |
       Orange: review ({counts.get('review', 0)})</p>
    <svg xmlns="http://www.w3.org/2000/svg" viewBox="{view_box}"
         role="img" aria-label="Walking profile decisions in central Clermont-Ferrand">
      <g transform="scale(1,-1)">{street_svg}</g>
    </svg>
    <p>Prototype assumptions; no route calculation or field validation. North is up.</p>
    <p>© OpenStreetMap contributors | ODbL 1.0 | Extract by Geofabrik</p>
  </div>
</body>
</html>
"""
preview_path.write_text(preview_html, encoding="utf-8")
display(HTML(preview_html))
print(f"Saved preview: {preview_path}")

Saved preview: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm/data/processed/clermont_walking_preview.html


## Bilan et limites

Nous disposons d'une première sélection explicable et de la liste des cas à examiner. Les règles ne traitent pas encore les barrières sur les nœuds, les restrictions portées par des relations, les restrictions de sens piéton en détail ou les horaires. Les tags directionnels et horaires connus sont laissés à examiner ; ce n'est pas un moteur de routage complet.

Les trottoirs et traversées peuvent être incomplets dans OSM. La présence d'un escalier ou d'une pente peut aussi modifier le temps et l'accessibilité. Ces limites devront accompagner les résultats du prototype.

Les sorties sont `clermont_walking_audit.geojson`, `clermont_walking_ways.geojson` et `clermont_walking_preview.html`, dans `data/processed/`.

**Prochaine étape :** construire un graphe depuis les voies et les nœuds OSM retenus. Le GeoJSON découpé est un aperçu : il ne suffit pas à établir les connexions. Aucun temps de marche n'est calculé ici.